## Step 1 — Read and inspect Bronze

In [0]:
# this tell the pipeline that the label is in the data. We going to set it to be true for the batch streaming
REQUIRE_LABEL = True

In [0]:


from pyspark.sql import functions as F

bronze_table = ("predictivemaintenance_dev.bronze_data_not_quality_checked.streaming")

silver_table = ("predictivemaintenance_dev.silver_data.streaming")

bronze_df = spark.table(bronze_table)

print("Bronze records:", bronze_df.count())
print("Unique UDI:", bronze_df.select("UDI").distinct().count())

bronze_df.printSchema()
display(bronze_df.limit(10))





## Step 2 — Define our data-quality rules

In [0]:
def check_data_quality(df, require_label=False):

    df = df.withColumn(
        "Type",
        F.upper(F.trim(F.col("Type")))
    )

    feature_valid = (
        F.col("UDI").isNotNull()
        & F.col("Type").isin("L", "M", "H")
        & (F.col("Air_temperature_K") > 0)
        & (F.col("Process_temperature_K") > 0)
        & (F.col("Rotational_speed_rpm") > 0)
        & (F.col("Torque_Nm") >= 0)
        & (F.col("Tool_wear_min") >= 0)
    )

    if require_label:
        label_valid = (
            F.col("Machine_failure").isNotNull()
            & F.col("Machine_failure").isin(0, 1)
        )

        quality_valid = feature_valid & label_valid

    else:
        quality_valid = feature_valid

    return df.withColumn(
        "_quality_pass",
        F.coalesce(quality_valid, F.lit(False))
    )


## Step 3 — Test the rules

In [0]:
bronze_checked = check_data_quality(
    bronze_df,
    require_label=REQUIRE_LABEL
)

display(
    bronze_checked
    .groupBy("_quality_pass")
    .count()
)

# Check duplicate records. Don't put this in the function above because the function above only check for each stream that arrives. If the same UDI is original in the table, it will not be caught. But doing below check all the table so it will be cut.
duplicates = (
    bronze_df
    .groupBy("UDI")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate UDIs:", duplicates.count())


## Step 4 — Create our streaming Silver table

In [0]:

#A checkpoint is how Spark remembers its progress.
# We put it in a volme because a Unity Catalog schema is not a file storage directory. We cannot put checkpoint files directly inside the schema. We need to create a Volume within that schema.
checkpoint_path = (
    "/Volumes/predictivemaintenance_dev/"
    "raw_data/landing/_checkpoints/bronze_to_silver"
)

# Read Bronze incrementally
bronze_stream_df = spark.readStream.table(bronze_table)

# Apply the same quality checks
silver_stream_df = (
    check_data_quality(
        bronze_stream_df,
        require_label=REQUIRE_LABEL
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

# Write incrementally into Silver
query = (
    silver_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

query.awaitTermination()



## Step 5 — Verify Silver

In [0]:

silver_df = spark.table(silver_table)

print("Bronze total:", bronze_df.count())
print("Silver total:", silver_df.count())

display(
    silver_df
    .groupBy("_quality_pass")
    .count()
)

display(silver_df.limit(10))
